## Memoria contigua vs memoria dispersa

**STACK (Pila - Ordenado y Rápido)**: 
* Aquí viven las variables locales (como int x, Node* head).
* La memoria se limpia sola cuando la función termina.
* En nuestra lista: Solo el puntero inicial (head) vive aquí.

**HEAP (Montón )**:

* Aquí viven los objetos creados con new.
* La memoria NO se limpia sola. (¡Peligro de Memory Leaks!).
* En nuestra lista: Todos los nodos con los datos viven aquí, dispersos.

Ejecute el la siguiente celda para observar la diferencia entre la memoria contigua y la memoria dispersa. Note cómo las direcciones de memoria del Arreglo son predecibles (se llevan 4 bytes), mientras que las variables en el Heap están 'donde caigan'

In [1]:
#include <iostream>

// --- ESCENARIO 1: EL ARREGLO (STACK) ---
std::cout << "=== MEMORIA CONTIGUA (ARRAY) ===\n";
int arr[3] = {10, 20, 30};
    
std::cout << "Direccion arr[0]: " << &arr[0] << "\n";
std::cout << "Direccion arr[1]: " << &arr[1] << " (Diferencia de 4 bytes exactos)\n";
std::cout << "Direccion arr[2]: " << &arr[2] << "\n\n";

// --- ESCENARIO 2: NODOS SIMULADOS (HEAP) ---
std::cout << "=== MEMORIA DISPERSA (SIMULACION LISTA) ===\n";
// Usamos 'new' para pedir memoria en el Heap
int* nodo1 = new int(10);
int* nodo2 = new int(20);
int* nodo3 = new int(30);

std::cout << "Direccion nodo1 (Heap): " << nodo1 << "\n";
std::cout << "Direccion nodo2 (Heap): " << nodo2 << " (¡Salto impredecible!)\n";
std::cout << "Direccion nodo3 (Heap): " << nodo3 << "\n";

// Limpieza manual (Regla de oro del Heap)
// Cada new le corresponde un delete
std::cout << "=== LIMPIEZA DE LA MEMORIA ===\n";
delete nodo1;
delete nodo2;
delete nodo3;

=== MEMORIA CONTIGUA (ARRAY) ===
Direccion arr[0]: 0xffff8d840000
Direccion arr[1]: 0xffff8d840004 (Diferencia de 4 bytes exactos)
Direccion arr[2]: 0xffff8d840008

=== MEMORIA DISPERSA (SIMULACION LISTA) ===
Direccion nodo1 (Heap): 0xaaaaebadf7b0
Direccion nodo2 (Heap): 0xaaaaec2b87b0 (¡Salto impredecible!)
Direccion nodo3 (Heap): 0xaaaaec2a6310
=== LIMPIEZA DE LA MEMORIA ===


## La Unidad Fundamental: `struct Node`
El nodo es el contenedor atómico. Técnicamente, es una estructura que reside en el Heap y contiene dos miembros:
* **Carga útil (Payload)**: El dato de tipo genérico T.
* **Puntero de enlace (Link)**: La dirección de memoria del siguiente nodo (next).

Utilizaremos `struct` en lugar de class para el nodo porque sus miembros deben ser accesibles directamente por la clase SinglyLL (friendship implícito). También vamos a inclir un constructor por movimiento.

### ¿Por qué es útil un Constructor por Movimiento en el Nodo?
Imagina que nuestra lista no guarda simples enteros (int), sino objetos pesados, como una `std::string` muy larga, un `std::vector` con mil datos, o una imagen.

* **Sin movimiento (Copia)**: Si haces `Node(T val)`, el compilador duplica todo el objeto val para guardarlo en data. Si val es una imagen de 10MB, acabas de gastar otros 10MB y tiempo de CPU copiando píxel por píxel.

* **Con movimiento (`T&&`)**: Le decimos al compilador: "Ese objeto val es temporal, nadie más lo va a usar. No lo copies, róbale las entrañas". El nodo se apropia de los recursos internos (punteros) del objeto original instantáneamente.

**nota**: A partir de este punto no ejecute las celdas

## La Clase Gestora: SinglyLL
La clase `SinglyLL` actúa como la interfaz de abstracción (ADT). El usuario de la clase no debe manipular nodos manualmente, sino invocar métodos.
Esta a clase requiere al menos dos atributos de estado interno:
* **`head`** (Cabeza): Un puntero que almacena la dirección del primer nodo. Es el único punto de entrada a la estructura.
* **`size`** (Tamaño): Un entero para rastrear la cardinalidad de la lista en $O(1)$ sin tener que recorrerla para contar.

## Inserción
Implementaremos tres estrategias de inserción:
* **`pushFront`**: Inserción rápida al inicio ($O(1)$).
* **`pushBack`**: Inserción al final con recorrido ($O(n)$).
* **`insertAt`**: Inserción quirúrgica en cualquier posición.

### Inserción al Inicio (`pushFront`)
Esta es la operación más eficiente de una lista enlazada. A diferencia de un Array, donde insertar al inicio obliga a desplazar todos los elementos a la derecha, aquí es instantáneo.

#### Lógica:

1. Creas un nodo nuevo.
2. El next del nuevo nodo apunta a donde apuntaba head (al antiguo primero).
3. Actualizas head para que apunte al nuevo nodo.

Añadimos este método dentro de la sección public. Observa el uso de std::move para aprovechar tu optimización anterior.

## Inserción al Final (`pushBack`)

Para poner algo al final, debemos recorrer toda la lista preguntando "¿eres tú el último?" hasta encontrar el nodo cuyo next sea `nullptr`.

**Caso Especial**: Si la lista está vacía (`head == nullptr`), el `pushBack` actúa igual que pushFront.

## Inserción Arbitraria (insertAt) - La microcirugía
Esta es la operación más delicada. Para insertar en el índice *k*, necesitamos detenernos en el nodo *k-1* (el nodo previo) para realizar el "puenteo".

**Riesgo**: Si rompemos el enlace antes de conectar el nuevo nodo, perdemos el resto de la lista.

## Eliminar al Inicio (`popFront`) - $O(1)$
Es lo opuesto a pushFront.
1. Guardamos una referencia temporal al nodo que vamos a matar.
2. Movemos head al siguiente nodo.
3. Aplicamos delete al temporal.

## Eliminar al Final (popBack) - $O(n)$
El problema aquí es que head no sabe quién es el penúltimo.
1. Si hay 0 o 1 nodo, es trivial (usamos `popFront`).
2. Si hay más, recorremos hasta llegar al penúltimo nodo (`current->next->next == nullptr`).
3. Borramos el último y ponemos el next del penúltimo en `nullptr`.

## Eliminar Arbitrario (`removeAt`) - $O(n)$
Requiere la técnica de "puenteo" (bridging).
1. Vamos al nodo previo (index - 1).
2. Guardamos el nodo a borrar (toDelete).
3. Conectamos el previo con el siguiente del que borraremos.
4. `delete toDelete`.

## El Destructor y `clear()`
A diferencia de Java o Python, aquí no hay Garbage Collector. Si cerramos el programa sin borrar los nodos, esa memoria queda ocupada hasta reiniciar el PC (en sistemas antiguos) o el SO la reclama.

**Estrategia**: Crear un método clear() que llame a popFront() repetidamente hasta que la lista esté vacía.